# Wnioskowanie regułowe w ocenie ryzyka transakcji kartowych

Notebook do projektu o wykrywaniu oszustw w transakcjach kartowych (zadania G1–G3, B1, P1). Z cech transakcji system wyprowadza poziom ryzyka i zalecenie dla analityka: zatwierdzić, eskalować albo zablokować do weryfikacji.

Cykl wnioskowania w przód: fakty, dopasowanie reguł (agenda), rozwiązanie konfliktu, wykonanie reguły, nowe fakty wraz z zapisem, skąd się wzięły.

## Miejsce w projekcie

Reguły uzupełniają modele z karty projektu (Isolation Forest, Random Forest): pokazują, dlaczego dana transakcja jest podejrzana. Decyzję zawsze podejmuje analityk.

Fakty wejściowe odpowiadają cechom IEEE-CIS, ale to założenie: kolumny są zanonimizowane, a progi (np. co znaczy wysoka kwota) trzeba by wyznaczyć na danych. Reguły są przykładowe i nie były uczone z danych.

| Fakt wejściowy | Odpowiednik w IEEE-CIS (założenie) |
| --- | --- |
| wysoka_kwota | TransactionAmt powyżej progu |
| kwota_nietypowa_dla_karty | TransactionAmt odbiega od zwykłych kwot karty (card1–card6) |
| duza_odleglosc_adresow | dist1–dist2, addr1 |
| inny_kraj | addr2 inny niż zwykle |
| rozne_domeny_email | P_emaildomain inna niż R_emaildomain |
| nowe_urzadzenie | DeviceType, DeviceInfo niewidziane wcześniej |
| wysokie_liczniki_c | C1–C14 |
| krotki_odstep_od_poprzedniej | D1–D15 |
| pierwsza_transakcja_karty | D1–D15 (brak historii) |
| flagi_m_niezgodne | M1–M9 = F |
| transakcja_nocna | TransactionDT (pora dnia) |
| wiele_malych_transakcji | niska TransactionAmt przy wysokich licznikach C |
| stala_historia_karty | D1–D15 (długa historia) |
| tozsamosc_zweryfikowana | id_01–id_38 |

## G1. Baza wiedzy

Reguły tworzą łańcuchy: cechy, anomalia, ryzyko, zalecenie. R15 jest celowo konkurencyjna wobec R7 (ta sama przesłanka, inne zalecenie) i służy do pokazania strategii konfliktu.

In [ ]:
# Baza wiedzy: jeśli wszystkie przesłanki są spełnione, to wniosek
REGULY = [
    # anomalie z cech transakcji
    {'id': 'R1', 'if': {'wysoka_kwota', 'kwota_nietypowa_dla_karty'}, 'then': 'anomalia_kwoty', 'prio': 1},
    {'id': 'R2', 'if': {'duza_odleglosc_adresow', 'inny_kraj'}, 'then': 'anomalia_lokalizacji', 'prio': 1},
    {'id': 'R3', 'if': {'nowe_urzadzenie', 'rozne_domeny_email'}, 'then': 'anomalia_tozsamosci', 'prio': 1},
    {'id': 'R4', 'if': {'wysokie_liczniki_c', 'krotki_odstep_od_poprzedniej'}, 'then': 'anomalia_aktywnosci', 'prio': 1},
    # poziom ryzyka
    {'id': 'R5', 'if': {'anomalia_kwoty', 'anomalia_lokalizacji'}, 'then': 'wysokie_ryzyko', 'prio': 3},
    {'id': 'R6', 'if': {'anomalia_aktywnosci', 'anomalia_tozsamosci'}, 'then': 'wysokie_ryzyko', 'prio': 3},
    {'id': 'R8', 'if': {'flagi_m_niezgodne', 'pierwsza_transakcja_karty'}, 'then': 'srednie_ryzyko', 'prio': 2},
    {'id': 'R10', 'if': {'anomalia_lokalizacji', 'transakcja_nocna'}, 'then': 'srednie_ryzyko', 'prio': 2},
    {'id': 'R11', 'if': {'stala_historia_karty', 'tozsamosc_zweryfikowana'}, 'then': 'niskie_ryzyko', 'prio': 1},
    {'id': 'R13', 'if': {'wiele_malych_transakcji', 'pierwsza_transakcja_karty'}, 'then': 'podejrzenie_testowania_karty', 'prio': 2},
    # zalecenia dla analityka
    {'id': 'R7', 'if': {'wysokie_ryzyko'}, 'then': 'zalecenie_zablokuj_do_weryfikacji', 'prio': 1},
    {'id': 'R9', 'if': {'srednie_ryzyko'}, 'then': 'zalecenie_eskaluj_kontakt_z_klientem', 'prio': 1},
    {'id': 'R12', 'if': {'niskie_ryzyko'}, 'then': 'zalecenie_zatwierdz', 'prio': 1},
    {'id': 'R14', 'if': {'podejrzenie_testowania_karty'}, 'then': 'zalecenie_zablokuj_do_weryfikacji', 'prio': 1},
    # konkurencyjna do R7: ta sama przesłanka, inne zalecenie
    {'id': 'R15', 'if': {'wysokie_ryzyko', 'tozsamosc_zweryfikowana'}, 'then': 'zalecenie_eskaluj_kontakt_z_klientem', 'prio': 2},
]

# fakty, które mogą przyjść z danych transakcji (potrzebne do wykrywania martwych reguł)
WEJSCIA = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj',
           'nowe_urzadzenie', 'rozne_domeny_email', 'wysokie_liczniki_c', 'krotki_odstep_od_poprzedniej',
           'flagi_m_niezgodne', 'pierwsza_transakcja_karty', 'transakcja_nocna', 'stala_historia_karty',
           'tozsamosc_zweryfikowana', 'wiele_malych_transakcji'}


## G1. Wnioskowanie w przód

Cykl: agenda, wybór reguły według strategii, wykonanie, zapis uzasadnienia. `stop_na_zaleceniu=True` kończy wnioskowanie po pierwszym zaleceniu.

In [ ]:
def forward_chain(fakty, reguly, strategia='kolejnosc', stop_na_zaleceniu=False, verbose=False):
    # zwraca (fakty, powod), gdzie powod[fakt] = (id reguły, przesłanki)
    fakty = set(fakty)
    powod = {}
    krok = 0
    while True:
        # agenda: reguły z spełnionymi przesłankami, których wniosku jeszcze nie ma
        agenda = [r for r in reguly if r['if'] <= fakty and r['then'] not in fakty]
        if not agenda:
            break
        # rozwiązanie konfliktu
        if strategia == 'kolejnosc':
            r = agenda[0]
        elif strategia == 'priorytet':
            r = max(agenda, key=lambda x: x['prio'])   # przy remisie pierwsza z listy
        else:
            raise ValueError('nieznana strategia: ' + strategia)
        # wykonanie reguły i zapis, skąd wniosek
        krok += 1
        fakty.add(r['then'])
        powod[r['then']] = (r['id'], sorted(r['if']))
        if verbose:
            print('krok', krok, r['id'], sorted(r['if']), '->', r['then'], '| agenda:', [a['id'] for a in agenda])
        if stop_na_zaleceniu and r['then'].startswith('zalecenie_'):
            break
    return fakty, powod


def wyjasnij(fakt, powod, poziom=0):
    # drzewo: skąd wziął się fakt
    wciecie = '  ' * poziom
    if fakt not in powod:
        print(wciecie + '- ' + fakt + '  [fakt startowy]')
        return
    rid, przeslanki = powod[fakt]
    print(wciecie + '- ' + fakt + '  [z ' + rid + ': ' + str(przeslanki) + ']')
    for p in przeslanki:
        wyjasnij(p, powod, poziom + 1)


# przykład: nietypowa kwota i lokalizacja
wynik, _ = forward_chain({'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj'},
                         REGULY, verbose=True)
print(sorted(wynik))


## G2. Konflikty i wyjaśnianie „dlaczego?”

Ta sama transakcja ze zweryfikowaną tożsamością, dwie strategie. Przy `kolejnosc` wychodzi blokada (R7), przy `priorytet` eskalacja (R15).

In [ ]:
TRANSAKCJA = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj',
              'tozsamosc_zweryfikowana'}

for strat in ['kolejnosc', 'priorytet']:
    print('=== strategia:', strat, '===')
    fakty, powod = forward_chain(TRANSAKCJA, REGULY, strategia=strat, stop_na_zaleceniu=True, verbose=True)
    zalecenia = sorted(f for f in fakty if f.startswith('zalecenie_'))
    print('zalecenie:', zalecenia)
    print('--- dlaczego? ---')
    for z in zalecenia:
        wyjasnij(z, powod)
    print()


## Reguły martwe i konfliktowe

Martwa reguła ma przesłankę, której nic nie produkuje i która nie jest faktem wejściowym. Reguły konfliktowe mają zawierające się przesłanki i różne wnioski. Para R7–R15 jest celowa.

In [ ]:
def analiza_bazy(reguly, wejscia=WEJSCIA):
    wyprodukowane = set(r['then'] for r in reguly)
    martwe = [r['id'] for r in reguly if any(p not in wyprodukowane and p not in wejscia for p in r['if'])]
    konflikty = []
    for i, a in enumerate(reguly):
        for b in reguly[i + 1:]:
            if a['then'] != b['then'] and (a['if'] <= b['if'] or b['if'] <= a['if']):
                konflikty.append((a['id'], b['id']))
    return martwe, konflikty

martwe, konflikty = analiza_bazy(REGULY)
print('martwe reguły:', martwe)
print('konfliktowe pary:', konflikty)

# sprawdzenie, czy martwa reguła zostanie wykryta
test = REGULY + [{'id': 'R99', 'if': {'cecha_spoza_zbioru'}, 'then': 'zalecenie_zatwierdz', 'prio': 1}]
print('z R99 -> martwe:', analiza_bazy(test)[0])


## G3. Testy

`musi` to fakty, które muszą wyjść, `nie_moze` to fakty, które nie mogą wyjść. S6 pokazuje, że sama wysoka kwota nie daje blokady.

In [ ]:
SCENARIUSZE = [
    {'nazwa': 'S1 nietypowa kwota + lokalizacja -> blokada', 'fakty': {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj'},
     'musi': {'wysokie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'},
     'nie_moze': {'zalecenie_eskaluj_kontakt_z_klientem', 'zalecenie_zatwierdz'}},
    {'nazwa': 'S2 aktywność + tożsamość -> blokada', 'fakty': {'wysokie_liczniki_c', 'krotki_odstep_od_poprzedniej', 'nowe_urzadzenie', 'rozne_domeny_email'},
     'musi': {'anomalia_aktywnosci', 'anomalia_tozsamosci', 'wysokie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'}, 'nie_moze': set()},
    {'nazwa': 'S3 niezgodne flagi M + nowa karta -> eskalacja', 'fakty': {'flagi_m_niezgodne', 'pierwsza_transakcja_karty'},
     'musi': {'srednie_ryzyko', 'zalecenie_eskaluj_kontakt_z_klientem'},
     'nie_moze': {'wysokie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'}},
    {'nazwa': 'S4 stała historia + zweryfikowana tożsamość -> zatwierdź', 'fakty': {'stala_historia_karty', 'tozsamosc_zweryfikowana'},
     'musi': {'niskie_ryzyko', 'zalecenie_zatwierdz'},
     'nie_moze': {'wysokie_ryzyko', 'srednie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'}},
    {'nazwa': 'S5 testowanie karty -> blokada', 'fakty': {'wiele_malych_transakcji', 'pierwsza_transakcja_karty'},
     'musi': {'podejrzenie_testowania_karty', 'zalecenie_zablokuj_do_weryfikacji'}, 'nie_moze': {'zalecenie_zatwierdz'}},
    {'nazwa': 'S6 sama wysoka kwota -> brak blokady', 'fakty': {'wysoka_kwota'},
     'musi': set(), 'nie_moze': {'anomalia_kwoty', 'zalecenie_zablokuj_do_weryfikacji'}},
    {'nazwa': 'S7 pusta lista faktów -> brak zaleceń', 'fakty': set(),
     'musi': set(), 'nie_moze': {'zalecenie_zablokuj_do_weryfikacji', 'zalecenie_eskaluj_kontakt_z_klientem', 'zalecenie_zatwierdz'}},
    {'nazwa': 'S8 anomalia lokalizacji w nocy -> eskalacja', 'fakty': {'duza_odleglosc_adresow', 'inny_kraj', 'transakcja_nocna'},
     'musi': {'anomalia_lokalizacji', 'srednie_ryzyko', 'zalecenie_eskaluj_kontakt_z_klientem'}, 'nie_moze': {'wysokie_ryzyko'}},
    {'nazwa': 'S9 sama anomalia tożsamości -> brak wysokiego ryzyka', 'fakty': {'nowe_urzadzenie', 'rozne_domeny_email'},
     'musi': {'anomalia_tozsamosci'}, 'nie_moze': {'wysokie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'}},
    {'nazwa': 'S10 sama nowa karta -> brak testowania karty', 'fakty': {'pierwsza_transakcja_karty'},
     'musi': set(), 'nie_moze': {'podejrzenie_testowania_karty', 'srednie_ryzyko'}},
    {'nazwa': 'S11 konflikt, kolejność -> blokada', 'fakty': TRANSAKCJA, 'strategia': 'kolejnosc', 'stop': True,
     'musi': {'zalecenie_zablokuj_do_weryfikacji'}, 'nie_moze': {'zalecenie_eskaluj_kontakt_z_klientem'}},
    {'nazwa': 'S12 konflikt, priorytet -> eskalacja', 'fakty': TRANSAKCJA, 'strategia': 'priorytet', 'stop': True,
     'musi': {'zalecenie_eskaluj_kontakt_z_klientem'}, 'nie_moze': {'zalecenie_zablokuj_do_weryfikacji'}},
]

def uruchom_testy(scenariusze, reguly=REGULY):
    zaliczone = 0
    for s in scenariusze:
        fakty, _ = forward_chain(s['fakty'], reguly, strategia=s.get('strategia', 'kolejnosc'),
                                 stop_na_zaleceniu=s.get('stop', False))
        ok = s['musi'] <= fakty and not (s['nie_moze'] & fakty)
        zaliczone += ok
        print('OK   ' if ok else 'BŁĄD ', s['nazwa'])
    print('Wynik:', zaliczone, '/', len(scenariusze))
    return zaliczone == len(scenariusze)

assert uruchom_testy(SCENARIUSZE), 'Są niezaliczone testy'


## B1. Wnioskowanie wstecz

Tu zaczynamy od celu (zalecenia) i sprawdzamy, czy da się go udowodnić z faktów oraz czego brakuje. W przód stosujemy, gdy mamy komplet cech i chcemy wszystkie wnioski. Wstecz, gdy analityk ma hipotezę (np. testowanie karty) i chce wiedzieć, co jeszcze sprawdzić.

In [ ]:
def backward_chain(cel, reguly, fakty, glebokosc=0, odwiedzone=frozenset()):
    # zwraca (czy udowodniony, ślad); cykle są odrzucane
    fakty = set(fakty)
    if cel in fakty:
        return True, [(glebokosc, cel, 'fakt')]
    if cel in odwiedzone:
        return False, [(glebokosc, cel, 'cykl - odrzucone')]
    odwiedzone = odwiedzone | {cel}
    slad_porazek = []
    for r in reguly:
        if r['then'] != cel:
            continue
        podslad = [(glebokosc, cel, 'z ' + r['id'] + ' ' + str(sorted(r['if'])))]
        ok = True
        for p in sorted(r['if']):
            udowodniony, s = backward_chain(p, reguly, fakty, glebokosc + 1, odwiedzone)
            podslad += s
            if not udowodniony:
                ok = False
                break
        if ok:
            return True, podslad
        slad_porazek += podslad + [(glebokosc, cel, r['id'] + ' odrzucona')]
    if not slad_porazek:
        slad_porazek = [(glebokosc, cel, 'brak reguły i brak faktu')]
    return False, slad_porazek


def pokaz_slad(slad):
    for glebokosc, cel, opis in slad:
        print('  ' * glebokosc + '- ' + cel + '  [' + opis + ']')


def brakujace_fakty(cel, reguly, fakty, odwiedzone=frozenset()):
    # najmniejszy zbiór brakujących faktów wejściowych potrzebny do celu
    if cel in fakty or cel in odwiedzone:
        return set()
    regs = [r for r in reguly if r['then'] == cel]
    if not regs:
        return {cel}
    kandydaci = []
    for r in regs:
        braki = set()
        for p in r['if']:
            braki |= brakujace_fakty(p, reguly, fakty, odwiedzone | {cel})
        kandydaci.append(braki)
    return min(kandydaci, key=len)


In [ ]:
# transakcja z nietypową kwotą i dużą odległością adresów, bez informacji o kraju
FAKTY_B1 = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow'}

for cel in ['anomalia_kwoty', 'zalecenie_zablokuj_do_weryfikacji', 'zalecenie_zatwierdz']:
    ok, slad = backward_chain(cel, REGULY, FAKTY_B1)
    print('CEL:', cel, '| udowodniony:', ok)
    pokaz_slad(slad)
    print('brakujące fakty:', sorted(brakujace_fakty(cel, REGULY, FAKTY_B1)))
    print()


In [ ]:
# kontrola: wstecz i w przód muszą dawać ten sam wynik dla każdego faktu w każdym scenariuszu
cele = sorted({r['then'] for r in REGULY} | WEJSCIA)
for s in SCENARIUSZE:
    pelne, _ = forward_chain(s['fakty'], REGULY)
    for cel in cele:
        ok_wstecz, _ = backward_chain(cel, REGULY, s['fakty'])
        assert ok_wstecz == (cel in pelne), (s['nazwa'], cel)
print('Zgodność wstecz/w przód: OK dla', len(SCENARIUSZE), 'scenariuszy i', len(cele), 'faktów')


## P1. Reguły jako guardrails dla LLM

Asystent LLM mógłby podpowiadać analitykowi akcję. Zanim ją przyjmiemy, reguły sprawdzają cztery rzeczy: czy fakty wejściowe są znane, czy akcja nie jest zakazana, czy wynika z cech transakcji (wnioskowanie w przód) oraz czy jest zatwierdzenie analityka. Propozycja modelu jest tu symulowana słownikiem, więc kod działa bez API.

In [ ]:
ZAKAZANE_AKCJE = {'trwale_zamknij_konto', 'usun_historie_transakcji', 'zablokuj_bez_weryfikacji_analityka'}
AKCJA_NA_ZALECENIE = {
    'zablokuj_do_weryfikacji': 'zalecenie_zablokuj_do_weryfikacji',
    'eskaluj_kontakt_z_klientem': 'zalecenie_eskaluj_kontakt_z_klientem',
    'zatwierdz': 'zalecenie_zatwierdz',
}
WYMAGANE_POLA = {akcja: ['zatwierdzenie_analityka'] for akcja in AKCJA_NA_ZALECENIE}


def guardrail(propozycja, fakty_transakcji, reguly=REGULY):
    # zwraca (czy dozwolona, lista powodów odrzucenia)
    powody = []
    # nieznane fakty wejściowe
    obce = set(fakty_transakcji) - WEJSCIA
    if obce:
        powody.append('nieznane fakty wejściowe: ' + str(sorted(obce)))
    akcja = propozycja.get('akcja')
    if akcja in ZAKAZANE_AKCJE:
        powody.append('akcja zakazana: ' + akcja)
    elif akcja not in AKCJA_NA_ZALECENIE:
        powody.append('akcja spoza bazy wiedzy: ' + str(akcja))
    else:
        # akcja musi wynikać z cech transakcji
        wyprowadzone, _ = forward_chain(fakty_transakcji, reguly)
        wymagane = AKCJA_NA_ZALECENIE[akcja]
        if wymagane not in wyprowadzone:
            powody.append('brak wsparcia w bazie: ' + wymagane + ' nie wynika z faktów')
    # zatwierdzenie analityka
    for pole in WYMAGANE_POLA.get(akcja, []):
        if not propozycja.get(pole):
            powody.append('brak wymaganego pola: ' + pole)
    return len(powody) == 0, powody


In [ ]:
FAKTY_WYSOKIE = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj'}
FAKTY_ZAUFANE = {'stala_historia_karty', 'tozsamosc_zweryfikowana'}

PRZYPADKI = [
    ('poprawna blokada', {'akcja': 'zablokuj_do_weryfikacji', 'zatwierdzenie_analityka': True}, FAKTY_WYSOKIE, True),
    ('poprawne zatwierdzenie', {'akcja': 'zatwierdz', 'zatwierdzenie_analityka': True}, FAKTY_ZAUFANE, True),
    ('zatwierdzenie ryzykownej transakcji', {'akcja': 'zatwierdz', 'zatwierdzenie_analityka': True}, FAKTY_WYSOKIE, False),
    ('brak zatwierdzenia analityka', {'akcja': 'zablokuj_do_weryfikacji'}, FAKTY_WYSOKIE, False),
    ('akcja zakazana', {'akcja': 'trwale_zamknij_konto', 'zatwierdzenie_analityka': True}, FAKTY_WYSOKIE, False),
    ('wstrzyknięty fakt', {'akcja': 'zablokuj_do_weryfikacji', 'zatwierdzenie_analityka': True},
     FAKTY_WYSOKIE | {'klient_vip_zatwierdzony'}, False),
    ('akcja spoza bazy', {'akcja': 'zwroc_srodki', 'zatwierdzenie_analityka': True}, FAKTY_WYSOKIE, False),
]

zaliczone = 0
for nazwa, prop, fakty, oczekiwane in PRZYPADKI:
    dozwolona, powody = guardrail(prop, fakty)
    zgodne = dozwolona == oczekiwane
    zaliczone += zgodne
    print('OK   ' if zgodne else 'BŁĄD ', nazwa, '->', 'dozwolona' if dozwolona else 'odrzucona', powody)
print('Wynik:', zaliczone, '/', len(PRZYPADKI))
assert zaliczone == len(PRZYPADKI)


## Wnioski

- Strategia konfliktu zmienia wynik przy R7 i R15: `kolejnosc` daje blokadę do weryfikacji, `priorytet` eskalację z kontaktem z klientem.
- W bazie nie ma martwych reguł. Jedyna para konfliktowa to R7–R15 i jest celowa. Testowa reguła R99 zostaje wykryta jako martwa.
- Wszystkie 12 testów przechodzi. Według S6 sama wysoka kwota nie blokuje transakcji, inaczej niż prosta reguła progowa z karty projektu.
- Reguły są przykładowe i nie były uczone z IEEE-CIS. Cech V1–V339 nie da się użyć w regułach, bo ich znaczenie jest niejawne.
- Ograniczenia: jedna reguła na cykl, przybliżone wykrywanie konfliktów, brak negacji i wag pewności.
- Dalej: fakty wyprowadzane z wyników modeli (np. wynik ryzyka powyżej progu) i porównanie jakości kolejki na PR AUC oraz precision@k.


## Użycie AI

Użyte narzędzie: Claude Code. Zastosowanie: pomoc przy pisaniu kodu, testów i opisów w tym notebooku.

Kontrola: wszystkie komórki uruchomione od początku do końca, testy przechodzą (12/12 oraz 7/7 dla guardrails), a wnioskowanie wstecz zgadza się z wnioskowaniem w przód. Ślady wnioskowania przejrzane ręcznie.
